## Phi-4 RAG Azure AI Search ഉപയോഗിച്ച്

Azure AI Search-നൊപ്പം Phi-4-mini ഉം Phi-4-multimodal ഉം Retrieval Augmented Generation (RAG) සඳහා എങ്ങനെ ഉപയോഗിക്കാമെന്ന് ഈ നോട്ട്‌ബുക്ക് കാണിക്കുന്നു. ഇത് ഏക-മോഡാലിറ്റി (ടെക്സ്റ്റ് മാത്രമുള്ള) മിഷ്ര-മോഡാലിറ്റി (ടെക്സ്റ്റും ചിത്രവും) ഇരുവിദമായ സാഹചര്യങ്ങളും ഉൾക്കൊള്ളുന്നു.

**ആവശ്യമുള്ളത്:**
*   Azure AI Search വെക്റ്റർ ഇൻഡക്സ് (ഒരു ഇൻഡക്സ് സൃഷ്‌ടിക്കാൻ [ഈ നിർദ്ദേശങ്ങൾ](https://learn.microsoft.com/azure/search/search-get-started-portal-import-vectors?tabs=sample-data-storage%2Cmodel-aoai%2Cconnect-data-storage) പാലിക്കുക)
*   Microsoft Foundry-യിൽ Phi-4-mini അല്ലെങ്കിൽ Phi-4-multimodal എൻഡ്‌പോയിന്റുകൾ വിന്യസിച്ചിരിക്കുന്നത്


In [ ]:
! pip install azure-ai-inference azure-search-documents

### Phi-4-mini ഉപയോഗിച്ച് ടെക്സ്റ്റ് മാത്രം RAG

ഈ വിഭാഗം Phi-4-mini നെ ഒരു ചാറ്റ് പൂർത്തീകരണ മാതൃകയായി ഉപയോഗിച്ച് RAG ൽ ടെക്സ്റ്റ് മാത്രമാണ് ഇൻപുട്ടായി ഉപയോഗിക്കുന്നത് എങ്ങനെ ചെയ്യാമെന്ന് കാണിക്കുന്നു. ഇതിൽ Microsoft Foundry Inference-ഉം Azure AI Search-ഉം ബന്ധിപ്പിച്ച്, പ്രസക്തമായ ഡോക്യുമെന്റുകൾ ലഭ്യമാക്കി, ലഭിച്ച संदർഭം ഉപയോഗിച്ച് ഒരു ഉത്തരം സൃഷ്ടിക്കുന്നു.


In [ ]:
# Step 1: Connect to Microsoft Foundry Inference & Azure AI Search
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], # Phi-4-mini endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 10):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query,vector_queries=[vector_query],select=["content"],top=top)
    return [doc["content"] for doc in results]

# Step 3: Generate answer using RAG!
def generate_rag_response(query: str):
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    response = chat_client.complete(messages=[UserMessage(content=prompt)])
    return response.choices[0].message.content

# Example usage
user_query = "What is the capital of France?"
answer = generate_rag_response(user_query)
print(f"Q: {user_query}\nA: {answer}")


### Phi-4-multimodal ഉപയോഗിച്ച് മൾട്ടി-മോടൽ RAG

ഈ വിഭാഗം Phi-4-multimodal നെ RAG ന് ചാറ്റ് കമ്പ്ലീഷൻ മോഡലായി ഉപയോഗിക്കുന്നത് എങ്ങനെ എന്നു കാണിക്കുന്നു, ഇത് ടെക്സ്റ്റും ഇമേജ് ഇൻപുട്ടും ഉൾപ്പെടുത്തുന്നു. ഇത് Azure AI Inference-നും Azure AI Search-നും ബന്ധിപ്പിക്കുന്നത്, അനുയോജ്യമായ ഡോക്യുമെന്റുകൾ തിരികെ കൊണ്ടുവരുന്നത്, മൾട്ടി-മോടൽ പ്രതികരണം സൃഷ്ടിക്കുന്നതോ എന്നിവ ഉൾക്കൊള്ളുന്നു.

**കുറിപ്പ്:** നിങ്ങളുടെ Azure AI Search ഇൻഡക്സിൽ `text_vector` மற்றும் `image_vector` ഫീൽഡുകൾ ഇരുവരും ഉണ്ടെങ്കിൽ, നിങ്ങൾ മൾട്ടി-വെക്ടർ ക്വറിയും നടത്താവുന്നതാണ്.


In [ ]:
# Step 1: Connect to Azure AI Inference & Azure AI Search
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery


chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], #Phi-4-multimodal serverless endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 3):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query, vector_queries=[vector_query], select=["content"], top=top)    
    return [doc["content"] for doc in results]

## Example for Muli-modal search if you have a text_vector AND image_vector field in your vector_index
## NOTE, image vectorization is in preview at the time of writing this code, please use azure-search-documents pypi version >11.6.0b6 
# def retrieve_documents_multimodal(query: str, image_url: str, top: int = 3):
#     text_vector_query = VectorizableTextQuery(
#         text=query,
#         k_nearest_neighbors=top,
#         fields="text_vector",
#         weight=0.5  # Adjust weight as needed
#     )
#     image_vector_query = VectorizableImageUrlQuery(
#         url=image_url,
#         k_nearest_neighbors=top,
#         fields="image_vector",
#         weight=0.5  # Adjust weight as needed
#     )

#     results = search_client.search(
#         search_text=query,  
#         vector_queries=[text_vector_query, image_vector_query],
#         select=["content"],
#         top=top
#     )
#     return [doc["content"] for doc in results]


# Step 3: Generate a multimodal RAG-based answer using retrieved text and an image input
def generate_multimodal_rag_response(query: str, image_url: str):
    # Retrieve text context from search
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)

    # Build a prompt that combines the retrieved context with the user query
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    # Create a chat request that includes both text and image input
    response = chat_client.complete(
        messages=[
            SystemMessage(content="You are a helpful assistant that can process both text and images."),
            UserMessage(
                content=[
                    TextContentItem(text=prompt),
                    ImageContentItem(image_url=ImageUrl(url=image_url, detail=ImageDetailLevel.HIGH)),
                ]
            ),
        ]
    )
    return response.choices[0].message.content

# Example usage
user_query = "Can you search for similar items to this shoe?"
sample_image_url = "https://images.unsplash.com/photo-1542291026-7eec264c27ff?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3"
answer = generate_multimodal_rag_response(user_query, sample_image_url)
print(f"Q: {user_query}\nA: {answer}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ഡിസ്‌ക്ലെയിമർ**:  
ഈ പ്രമാണം AI തർജ്ജമാസേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് തർജ്ജമ ചെയ്‌തതാണ്. നാം കൃത്യത നേടുവാൻ ശ്രമിച്ചിരുന്നുവെങ്കിലും, യാന്ത്രിക തർജ്ജമയിൽ പിശകുകൾ അല്ലെങ്കിൽ അശുദ്ധതകൾ ഉണ്ടാകാവുന്നുണ്ടെന്ന് ശ്രദ്ധിക്കുക. മൂല പ്രമാണം അതിന്റെ ജന്മഭൂമിയിലെ ഭാഷയിൽ അതോറിറ്റേറ്റീവ് ഉറവിടമായി കാണേണ്ടതാണ്. അത്യന്താപേക്ഷിത വിവരങ്ങൾക്കായി പ്രൊഫഷണൽ മാനവ തർജ്ജമ ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ തർജ്മ മൂലം അവർ(cuda കുട്ടികൾ) ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾക്കും വ്യാഖ്യാനക്കുറവുകൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
